# Retriever, LCEL 체인의 표준 인터페이스
- 이전에는 Chroma vectorDB 에 문서를 저장하고 `similarity_search()` 로 직접 검색했음
- 이번에는 대한민국 헌법 PDF를 로드해 vectorstore 를 만들고 **Retriever** 인터페이스로 검색 전략을 바꿔 봄
- Retriever 는 `invoke(질문) -> list[Document]` 형태의 표준 검색 인터페이스


## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [2]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-chroma langchain-openai langchain-text-splitters python-dotenv chromadb pypdf


## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## 2. Retriever 핵심 개념

| 개념 | 설명 |
|---|---|
| Loader | PDF, 웹, 텍스트 파일을 `Document` 목록으로 읽어옴 |
| Splitter | 긴 문서를 검색 가능한 작은 청크로 나눔 |
| VectorStore | 청크를 임베딩해 저장하고 직접 검색하는 저장소 |
| Retriever | 질문을 받아 관련 문서 목록을 반환하는 표준 인터페이스 |
| `search_kwargs` | `k`, `filter`, `score_threshold` 같은 검색 옵션 |
| `search_type` | `similarity`, `mmr`, `similarity_score_threshold` |
| LCEL 연결 | `retriever | format_docs` 처럼 체인 안에 삽입 |


## 3. 헌법 PDF 로드 + 청크 분할

- `PyPDFLoader(..., mode="page")` 는 PDF를 페이지 단위 `Document` 목록으로 읽음
- 각 문서의 metadata 에는 보통 `source`, `page` 같은 출처 정보가 들어감


In [4]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 사용자가 요청한 경로 형식: ../data/...
pdf_path = Path(r"data2/대한민국헌법(헌법)(제00010호)(19880225).pdf")

loader = PyPDFLoader(str(pdf_path), mode="page")
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
docs = text_splitter.split_documents(pages)

doc_ids = [f"constitution-{i:04d}" for i in range(len(docs))]

print(f"PDF 페이지 수: {len(pages)}")
print(f"검색용 청크 수: {len(docs)}")
print("첫 번째 청크 metadata:", docs[0].metadata)
print(docs[0].page_content[:300])

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_37580\575533902.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF 페이지 수: 14
검색용 청크 수: 52
첫 번째 청크 metadata: {'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2024-04-01T21:26:24+09:00', 'moddate': '2024-04-01T21:26:24+09:00', 'source': 'data2\\대한민국헌법(헌법)(제00010호)(19880225).pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}
법제처                                                            1                                                       국가법령정보센터
대한민국헌법
 
대한민국헌법
[시행 1988. 2. 25.] [헌법 제10호, 1987. 10. 29., 전부개정]
       전문
 
유구한 역사와 전통에 빛나는 우리 대한국민은 3ㆍ1운동으로 건립된 대한민국임시정부의 법통과 불의에 항거한
4ㆍ19민주이념을 계승하고, 조국의 민주개혁과 평화적 통일의 사명


## 4. Chroma vectorstore 만들기

- 헌법 청크를 OpenAI 임베딩으로 벡터화해 Chroma에 저장
- 수업 중 같은 셀을 반복 실행해도 ID가 중복되지 않도록 기존 collection을 삭제한 뒤 다시 만듦


In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
collection_name = "constitution_retriever"

# 반복 실행 대비: 같은 collection이 있으면 지우고 다시 생성
reset_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
)
reset_store.delete_collection()

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    ids=doc_ids,
    collection_name=collection_name,
)

print("저장 문서 수:", vectorstore._collection.count())


저장 문서 수: 52


## 5. 기본 retriever, `as_retriever()`
- `similarity_search()` 는 vectorstore의 메서드
- `as_retriever()` 를 사용하면 LCEL 체인에 넣기 쉬운 표준 retriever가 됨


In [6]:
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

results = retriever.invoke("국회의원의 의무는 무엇인가요?")
for doc in results:
    print(f"[page={doc.metadata.get('page')}] {doc.page_content[:180]}\n")

[page=4] ②국회의원은 국가이익을 우선하여 양심에 따라 직무를 행한다.
③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리ㆍ이
익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다.
 
제47조 ①국회의 정기회는 법률이 정하는 바에 의하여 매년 1회 집회되며, 국회의

[page=4] 법제처                                                            5                                                       국가법령정보센터
대한민국헌법
③국회의원의 선거구와 비례대표제 기타 선거에 관한 사항은 법률로 정한다.
 
제4

[page=5] 제61조 ①국회는 국정을 감사하거나 특정한 국정사안에 대하여 조사할 수 있으며, 이에 필요한 서류의 제출 또는 증인
의 출석과 증언이나 의견의 진술을 요구할 수 있다.
②국정감사 및 조사에 관한 절차 기타 필요한 사항은 법률로 정한다.
 
제62조 ①국무총리ㆍ국무위원 또는 정부위원은 국회나 그 위원회에 출석하여 국정처리상



## 6. 메타데이터 필터 retriever

- PDF 로더가 붙인 `page` metadata 를 이용해 특정 페이지 범위 안에서만 검색할 수 있음
- 페이지 번호는 0부터 시작


In [ ]:
page_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":3,
        "filter":{"page":{"$lte":3}}        # $lte : Less than or equl (작거나 같다)
    }
)

results = page_retriever.invoke("국회의원의 의무는 무엇인가요?")
for doc in results:
    print(f"[page={doc.metadata.get('page')}] {doc.page_content[:180]}\n")


[page=3] 한할 수 있으며, 제한하는 경우에도 자유와 권리의 본질적인 내용을 침해할 수 없다.
 
제38조 모든 국민은 법률이 정하는 바에 의하여 납세의 의무를 진다.
 
제39조 ①모든 국민은 법률이 정하는 바에 의하여 국방의 의무를 진다.
②누구든지 병역의무의 이행으로 인하여 불이익한 처우를 받지 아니한다.
 
       제3

[page=0] 제5조 ①대한민국은 국제평화의 유지에 노력하고 침략적 전쟁을 부인한다.
②국군은 국가의 안전보장과 국토방위의 신성한 의무를 수행함을 사명으로 하며, 그 정치적 중립성은 준수된다.
 
제6조 ①헌법에 의하여 체결ㆍ공포된 조약과 일반적으로 승인된 국제법규는 국내법과 같은 효력을 가진다.
②외국인은 국제법과 조약이 정하는 바에

[page=2] 제23조 ①모든 국민의 재산권은 보장된다. 그 내용과 한계는 법률로 정한다.
②재산권의 행사는 공공복리에 적합하도록 하여야 한다.
③공공필요에 의한 재산권의 수용ㆍ사용 또는 제한 및 그에 대한 보상은 법률로써 하되, 정당한 보상을 지급하여야
한다.
 
제24조 모든 국민은 법률이 정하는 바에 의하여 선거권을 가진다.
 




In [8]:
source_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":3,
        "filter":{"source":str(pdf_path)}       # 전체 vectorstore 중 source가 pdf_path와 같은 청크 안에서만 관련 문서를 찾음    
    }
)

results = source_retriever.invoke("국회의원의 의무는 무엇인가요?")
for doc in results:
    print(f"[page={doc.metadata.get('page')}] {doc.page_content[:180]}\n")


[page=4] ②국회의원은 국가이익을 우선하여 양심에 따라 직무를 행한다.
③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리ㆍ이
익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다.
 
제47조 ①국회의 정기회는 법률이 정하는 바에 의하여 매년 1회 집회되며, 국회의

[page=4] 법제처                                                            5                                                       국가법령정보센터
대한민국헌법
③국회의원의 선거구와 비례대표제 기타 선거에 관한 사항은 법률로 정한다.
 
제4

[page=5] 제61조 ①국회는 국정을 감사하거나 특정한 국정사안에 대하여 조사할 수 있으며, 이에 필요한 서류의 제출 또는 증인
의 출석과 증언이나 의견의 진술을 요구할 수 있다.
②국정감사 및 조사에 관한 절차 기타 필요한 사항은 법률로 정한다.
 
제62조 ①국무총리ㆍ국무위원 또는 정부위원은 국회나 그 위원회에 출석하여 국정처리상



## 7. MMR, 중복을 줄이는 다양성 검색

- MMR(Maximal Marginal Relevance)은 질문과 유사하면서도 서로 다른 문서를 골라줌
- 같은 표현의 청크가 반복될 때 유용

In [9]:
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,                  # 최종 반환 개수
        "fetch_k":10,           # 후보 풀(k 보다 크게)
        "lambda_mult" : 0.5     # 유사도와 다양성의 균형을 조절하는 값. 0에 가까울수록 다양성, 1에 가까울수록 유사도
    }
)

results = mmr_retriever.invoke("국민의 권리와 의무")
for doc in results:
    print(f"[page={doc.metadata.get('page')}] {doc.page_content[:180]}\n")


[page=1] 법제처                                                            2                                                       국가법령정보센터
대한민국헌법
 
       제2장 국민의 권리와 의무
 
제10조 모든 국민은 인간으로서의 

[page=3] 한할 수 있으며, 제한하는 경우에도 자유와 권리의 본질적인 내용을 침해할 수 없다.
 
제38조 모든 국민은 법률이 정하는 바에 의하여 납세의 의무를 진다.
 
제39조 ①모든 국민은 법률이 정하는 바에 의하여 국방의 의무를 진다.
②누구든지 병역의무의 이행으로 인하여 불이익한 처우를 받지 아니한다.
 
       제3

[page=1] 당하지 아니한다. 체포 또는 구속을 당한 자의 가족등 법률이 정하는 자에게는 그 이유와 일시ㆍ장소가 지체없이 통
지되어야 한다.
⑥누구든지 체포 또는 구속을 당한 때에는 적부의 심사를 법원에 청구할 권리를 가진다.
⑦피고인의 자백이 고문ㆍ폭행ㆍ협박ㆍ구속의 부당한 장기화 또는 기망 기타의 방법에 의하여 자의로 진술된 것이




## 8. Score threshold, 관련 없는 문서 줄이기

- 자료에 없는 질문인데도 억지로 top-k 문서를 넣으면 LLM이 그럴듯하게 답을 만들 수 있음
- 질문과 너무 동떨어진 청크를 LLM에 넣지 않도록 임계값 설정

> `similarity_score_threshold` 의 `score_threshold` 는 LangChain relevance score 기준입니다. 높을수록 더 엄격합니다.


In [10]:
threshold_retriever = vectorstore.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={
        "k":5,
        "score_threshold":0.35  # 관련도 점수가 0.35보다 낮으면 결과에서 제외함.
    }
)

question1 = "국회의원의 의무는 무엇인가요?"

results = threshold_retriever.invoke(question1)

print(f"\n질문: {question1}")
print(f"가져온 문서 수: {len(results)}")
for doc in results:
    print(f"  [page={doc.metadata.get('page')}] {doc.page_content[:120]}")


질문: 국회의원의 의무는 무엇인가요?
가져온 문서 수: 5
  [page=4] ②국회의원은 국가이익을 우선하여 양심에 따라 직무를 행한다.
③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리ㆍ이
익 또는 직위를 취득하거나 타인을 위하여 그 취
  [page=4] 법제처                                                            5                                                       국
  [page=5] 제61조 ①국회는 국정을 감사하거나 특정한 국정사안에 대하여 조사할 수 있으며, 이에 필요한 서류의 제출 또는 증인
의 출석과 증언이나 의견의 진술을 요구할 수 있다.
②국정감사 및 조사에 관한 절차 기타 필요한 사
  [page=5] ②국회는 의원의 자격을 심사하며, 의원을 징계할 수 있다.
③의원을 제명하려면 국회재적의원 3분의 2 이상의 찬성이 있어야 한다.
④제2항과 제3항의 처분에 대하여는 법원에 제소할 수 없다.
 
제65조 ①대통령ㆍ국
  [page=3] 한할 수 있으며, 제한하는 경우에도 자유와 권리의 본질적인 내용을 침해할 수 없다.
 
제38조 모든 국민은 법률이 정하는 바에 의하여 납세의 의무를 진다.
 
제39조 ①모든 국민은 법률이 정하는 바에 의하여 국방


In [11]:
question2 = "오늘 구내식당 메뉴는 무엇인가요?"

results2 = threshold_retriever.invoke(question2)

print(f"\n질문: {question2}")
print(f"가져온 문서 수: {len(results2)}")
for doc in results2:
    print(f"  [page={doc.metadata.get('page')}] {doc.page_content[:120]}")

c:\Users\playdata2\work_space(playdata)\SKN30_playdata\LLM\.venv\Lib\site-packages\langchain_core\vectorstores\base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='constitution-0033', metadata={'total_pages': 14, 'creator': 'PyPDF', 'page': 8, 'moddate': '2024-04-01T21:26:24+09:00', 'producer': 'iText 2.1.7 by 1T3XT', 'creationdate': '2024-04-01T21:26:24+09:00', 'source': 'data2\\대한민국헌법(헌법)(제00010호)(19880225).pdf', 'page_label': '9'}, page_content='6. 군사에 관한 중요사항\n7. 국회의 임시회 집회의 요구\n8. 영전수여\n9. 사면ㆍ감형과 복권\n10. 행정각부간의 권한의 획정\n11. 정부 안의 권한의 위임 또는 배정에 관한 기본계획\n12. 국정처리상황의 평가ㆍ분석\n13. 행정각부의 중요한 정책의 수립과 조정\n14. 정당해산의 제소\n15. 정부에 제출 또는 회부된 정부의 정책에 관계되는 청원의 심사\n16. 검찰총장ㆍ합동참모의장ㆍ각군참모총장ㆍ국립대학교총장ㆍ대사 기타 법률이 정한 공무원과 국영기업체관리자\n의 임명\n17. 기타 대통령ㆍ국무총리 또는 국무위원이 제출한 사항\n \n제90조 ①국정의 중요한 사항에 관한 대통령의 자문에 응하기 위하여 국가원로로 구성되는 국가원로자문회의를 둘 수\n있다.\n②국가원로자문회의의 의장은 직전대통령이 된다. 다만, 직전대통령이 없을 때에는 대통령이 지명한다.\n③국가원로자문회의의 조직ㆍ직무범위 기타 필요한 사항은 법률로 정한다.'), -0.12652861533179882), (Document(id=


질문: 오늘 구내식당 메뉴는 무엇인가요?
가져온 문서 수: 0


## 9. 검색 결과 포맷팅

- Retriever 는 `Document` 목록을 반환합니다. LLM 프롬프트에는 출처와 본문이 잘 보이도록 문자열로 바꿔 넣습니다.


In [12]:
from langchain_core.documents import Document

def format_docs(docs: list[Document]) -> str:
    if not docs:
        return "검색된 참고 자료가 없습니다."

    formatted = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata
        formatted.append(
            f"[{i}] source={meta.get('source')}, page={meta.get('page')}\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(formatted)


sample_docs = retriever.invoke("대통령의 임기")
print(format_docs(sample_docs))

[1] source=data2\대한민국헌법(헌법)(제00010호)(19880225).pdf, page=6
노력하여 대통령으로서의 직책을 성실히 수행할 것을 국민 앞에 엄숙히 선서합니다.”
 
제70조 대통령의 임기는 5년으로 하며, 중임할 수 없다.
 
제71조 대통령이 궐위되거나 사고로 인하여 직무를 수행할 수 없을 때에는 국무총리, 법률이 정한 국무위원의 순서로
그 권한을 대행한다.
 
제72조 대통령은 필요하다고 인정할 때에는 외교ㆍ국방ㆍ통일 기타 국가안위에 관한 중요정책을 국민투표에 붙일 수
있다.
 
제73조 대통령은 조약을 체결ㆍ비준하고, 외교사절을 신임ㆍ접수 또는 파견하며, 선전포고와 강화를 한다.
 
제74조 ①대통령은 헌법과 법률이 정하는 바에 의하여 국군을 통수한다.
②국군의 조직과 편성은 법률로 정한다.
 
제75조 대통령은 법률에서 구체적으로 범위를 정하여 위임받은 사항과 법률을 집행하기 위하여 필요한 사항에 관하여
대통령령을 발할 수 있다.
 
제76조 ①대통령은 내우ㆍ외환ㆍ천재ㆍ지변 또는 중대한 재정ㆍ경제상의 위기에 있어서 국가의 안전보장 또는 공공의

[2] source=data2\대한민국헌법(헌법)(제00010호)(19880225).pdf, page=6
법제처                                                            7                                                       국가법령정보센터
대한민국헌법
있어야 한다.
③탄핵소추의 의결을 받은 자는 탄핵심판이 있을 때까지 그 권한행사가 정지된다.
④탄핵결정은 공직으로부터 파면함에 그친다. 그러나, 이에 의하여 민사상이나 형사상의 책임이 면제되지는 아니한
다.
 
       제4장 정부
         제1절 대통령
 
제66조 ①대통령은 국가의 원수이며, 외국에 대하여 국가를 대표한다.
②대통령은 국가의 독립ㆍ영토의 보전ㆍ국가의 계속성과 헌법을 수호할 책무를 진다.
③대통령은 조국의 평화적 통일

## 10. LCEL RAG 체인에 retriever 연결


In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "너는 대한민국 헌법 QA Assistant다. 아래 참고 자료만 근거로 한국어로 답해라."
        "참고 자료에 없으면 '자료에서 확인할 수 없습니다.'라고 답해라"
        "가능하면 근거 페이지를 함께 언급하라."
    ),
    ("user", "참고 자료:\n{context}\n\n질문: {question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),  # 입력 질문을 그대로 question 변수에 넣겠다.
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)


## 11. 실행해보기


In [14]:
quesiton1 = '국회의원의 의무는 무엇인가?'
quesiton2 = '오늘 점심 메뉴는 무엇일까요?'

print(rag_chain.invoke(quesiton1))

국회의원의 의무는 다음과 같습니다.

1. 청렴의 의무가 있다. (제46조 ①)
2. 국가이익을 우선하여 양심에 따라 직무를 행한다. (제46조 ②, 제10조 ②)
3. 그 지위를 남용하여 국가·공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리·이익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다. (제10조 ③)

이 내용은 대한민국 헌법 제10조와 제46조에 근거하고 있습니다. (참고: [1] page 4, [2] page 4)


In [15]:
rag_chain.get_graph().print_ascii()

             +---------------------------------+          
             | Parallel<context,question>Input |          
             +---------------------------------+          
                    ***                ***                
                 ***                      ***             
               **                            ***          
+----------------------+                        **        
| VectorStoreRetriever |                         *        
+----------------------+                         *        
            *                                    *        
            *                                    *        
            *                                    *        
    +-------------+                       +-------------+ 
    | format_docs |                       | Passthrough | 
    +-------------+*                      +-------------+ 
                    ***                ***                
                       ***          ***                 

## 12. 정리

- `PyPDFLoader(..., mode="page")` 로 PDF를 페이지 단위 `Document`로 읽을 수 있습니다.
- `RecursiveCharacterTextSplitter` 로 페이지 문서를 검색용 청크로 나눕니다.
- `vectorstore.as_retriever()` 는 LCEL 체인에 넣기 좋은 표준 검색 인터페이스입니다.
- metadata 필터는 PDF의 `source`, `page` 같은 출처 정보로 제한할 수 있습니다.
- MMR 은 유사도와 다양성을 같이 고려해 중복 청크를 줄입니다.
- threshold retriever 는 관련도가 낮은 문서를 프롬프트에 넣지 않아 환각을 줄이는 데 도움됩니다.


## [실습]

1. 기본 retriever 의 `k` 값을 1, 3, 5 로 바꿔 `국회의원의 의무` 검색 결과를 비교합니다.
2. `filter={"page": {"$lte": 3}}` 조건으로 헌법 앞부분 질문만 검색합니다.
3. MMR 의 `lambda_mult` 를 0, 0.5, 1 로 바꿔 `국민의 권리와 의무` 결과 다양성을 비교합니다.
4. `score_threshold` 를 0.2, 0.35, 0.5 로 바꿔 자료 밖 질문의 반환 문서 수를 확인합니다.
5. `rag_chain` 에 기본 retriever, MMR retriever, threshold retriever 를 각각 연결해 답변 차이를 비교합니다.


In [ ]:
for k in [1, 3, 5]:
    page_retriever = vectorstore.as_retriever(
        search_kwargs={
            'k':k,
            'filter':{'page':{'$lte':3}}
        }
    )

    results_2 = page_retriever.invoke("국회의원의 의무는 무엇인가요?")
    print(f"k={k}, 검색 문서 수={len(results_2)}")
    for doc in results_2:
        print(f'[page={doc.metadata.get('page')}]{doc.page_content[:180]}\n')

In [ ]:
for sth in [0.2, 0.35, 0.5]:
    threshold_retriever_retriever = vectorstore.as_retriever(
        search_type='similarity_score_threshold',
        search_kwargs={
            'k':5,
            'score_threshold': sth
    }
    )
    question3 = "국민의 권리와 의무는 무엇인가요?"
    results_3 = mmr_retriever.invoke(question3)
    
    print(f"\n질문: {question3}, score_threshold={sth}, 가져온 문서 수={len(results_3)}")
    for doc in results_3:
        print(f'    [page={doc.metadata.get('page')}] {doc.page_content[:120]}')
